<a href="https://colab.research.google.com/github/upointx/Lohani-Face-Photo-Studio/blob/main/Lohani-Face-Photo-Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Light Install (Quota Saver 2026)
!pip install -q diffusers==0.30.2 transformers==4.41.2 accelerate xformers insightface onnxruntime-gpu gradio pillow opencv-python-headless
print("✅ Install done!")

In [ ]:

# @title 2. Drive Mount + Folders
from google.colab import drive
drive.mount('/content/drive')

import os
output_dir = '/content/drive/MyDrive/Lohani_Face_Studio'
os.makedirs(output_dir, exist_ok=True)
print(f"✅ Save folder: {output_dir}")

In [ ]:

# @title 3. Download Swapper + Load Face
import cv2
from insightface.app import FaceAnalysis
from insightface.model_zoo import get_model

!wget -q https://github.com/facefusion/facefusion-assets/releases/download/models/inswapper_128.onnx -O /content/inswapper_128.onnx

app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

face_ref_path = '/content/drive/MyDrive/Lohani/Face.jpg'  # তোমার ফাইল এখানে রাখো
if os.path.exists(face_ref_path):
    face_img = cv2.imread(face_ref_path)
    faces = app.get(face_img)
    face_reference = faces[0] if faces else None
    print("✅ Face loaded successfully!")
else:
    print("❌ Face.jpg না পাওয়া গেছে। Drive-এ Lohani/Face.jpg রাখো")

In [ ]:

# @title 4. Load Model (Realistic + Low VRAM)
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler, AutoencoderKL
import torch

model_id = "SG161222/Realistic_Vision_V6.0_B1_noVAE"
vae_id = "stabilityai/sd-vae-ft-mse-original"

vae = AutoencoderKL.from_pretrained(vae_id, torch_dtype=torch.float16)
pipe = StableDiffusionPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_xformers_memory_efficient_attention()
pipe.enable_model_cpu_offload()
print("✅ Model loaded (T4 optimized)")

In [ ]:
# @title 5. Swapper + Random Prompt System
swapper = get_model('/content/inswapper_128.onnx', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])

import random
import time
from PIL import Image
import gradio as gr

outfits = ["bold red satin backless gown", "sheer black lace bodysuit", "gold micro bikini", "leather catsuit", "transparent white silk dress", ...]  # তোমার আগের লিস্ট পেস্ট করো (১০+ আইটেম)
# (আমি সংক্ষেপে দিলাম, তুমি পুরো লিস্ট রাখো)

locations, poses, angles, lightings = [...]  # আগের মতো

def generate_images(num_images=4, custom_prompt=""):
    images = []
    for i in range(num_images):
        outfit = random.choice(outfits)
        loc = random.choice(locations)
        pose = random.choice(poses)
        angle = random.choice(angles)
        light = random.choice(lightings)

        prompt = f"ultra realistic full body photo of beautiful woman, {outfit}, {loc}, {pose}, {angle}, {light}, identical face to reference, natural skin, 8k, cinematic"
        if custom_prompt:
            prompt += f", {custom_prompt}"

        with torch.autocast("cuda", dtype=torch.float16):
            img = pipe(prompt, negative_prompt="deformed, blurry, bad anatomy, extra limbs", num_inference_steps=25, guidance_scale=7.5, height=768, width=512).images[0]

        # Face swap
        img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        gen_faces = app.get(img_cv)
        if gen_faces:
            res = swapper.get(img_cv, gen_faces[0], face_reference, paste_back=True)
            final = Image.fromarray(cv2.cvtColor(res, cv2.COLOR_BGR2RGB))
        else:
            final = img

        # Save
        save_path = os.path.join(output_dir, f"photo_{time.strftime('%Y%m%d_%H%M%S')}_{i+1}.png")
        final.save(save_path)
        images.append(final)

    return images

# Gradio UI
with gr.Blocks(title="Lohani Face Photo Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌟 Lohani Face Photo Studio\n**Same Face • Unlimited Styles**")
    with gr.Row():
        with gr.Column():
            num = gr.Slider(1, 8, value=4, step=1, label="কয়টা ছবি চাও?")
            custom = gr.Textbox(label="Extra Prompt (optional)", placeholder="rainy night, smiling, red lipstick")
            btn = gr.Button("🚀 Generate Now", variant="primary", size="large")
        with gr.Column():
            gallery = gr.Gallery(label="Generated Photos", height=600)

    btn.click(fn=generate_images, inputs=[num, custom], outputs=gallery)

demo.launch(share=True, debug=True)

In [ ]:
# Test one image
print("Test generation started...")
imgs = generate_images(1)
imgs[0].show()